### Create event frames of event data

In [8]:
import os
import glob
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm

In [9]:
WIDTH = 640
HEIGHT = 480

In [10]:
recording_dir = "//172.16.222.2/tmp/Nora/traffic_flow_data/week_34/box_1/2024_08_27_16_55_29_recordings"
clip_dir = os.path.join(recording_dir, "event_data")
timestamp_path = "//172.16.222.2/EBVrecordings/2024-TrafficMonitoring/data_collection/week_34/box_1/2024_08_27_16_55_29_recordings/timestamps.csv"
output_dir = os.path.join(recording_dir, "event_frames")
os.makedirs(output_dir, exist_ok=True)

In [11]:
df_timestamps = pd.read_csv(timestamp_path)

SHIFT = df_timestamps.index[df_timestamps["event_time"] != 0][0]

time_windows1 = df_timestamps.iloc[SHIFT:, 0].to_numpy().astype(float)
time_windows = time_windows1 - time_windows1[0]

In [12]:
def aggregate_events_to_frames(events, time_windows, width, height):
    n_frames = len(time_windows) - 1
    frames = np.zeros((n_frames, height, width), dtype=np.uint16)

    xs = events["x"]
    ys = events["y"]
    ts = events["t"]

    event_index = 0
    n_events = len(ts)

    for i in tqdm(range(n_frames), leave=False):
        start_t = time_windows[i]
        end_t = time_windows[i + 1]

        while event_index < n_events and ts[event_index] < end_t:
            if ts[event_index] >= start_t:
                x = int(xs[event_index])
                y = int(ys[event_index])
                frames[i, y, x] = 1
            event_index += 1

    return frames

In [13]:
def process_event_clip_to_frames(clip_path, time_windows, output_dir, width, height):
    clip_name = os.path.basename(clip_path).replace(".h5", "")
    out_path = os.path.join(output_dir, f"{clip_name}_frames.h5")

    with h5py.File(clip_path, "r") as f:
        events = f["events"][:]
        start_frame = int(f["start_frame"][()])
        end_frame = int(f["end_frame"][()])
        start_ts = int(f["start_ts"][()])
        end_ts = int(f["end_ts"][()])

    clip_time_windows = time_windows[start_frame:end_frame + 1]

    if len(clip_time_windows) < 2:
        print(f"Skipping {clip_name}: not enough time windows")
        return

    frames = aggregate_events_to_frames(
        events=events,
        time_windows=clip_time_windows,
        width=width,
        height=height
    )

    with h5py.File(out_path, "w") as f:
        f.create_dataset(
            "frames",
            data=frames,
            compression="gzip",
            chunks=(1, height, width),
        )
        f.create_dataset("time_windows", data=clip_time_windows)
        f.create_dataset("start_frame", data=start_frame)
        f.create_dataset("end_frame", data=end_frame)
        f.create_dataset("start_ts", data=start_ts)
        f.create_dataset("end_ts", data=end_ts)

    print("Wrote:", out_path)

In [ ]:
clip_paths = sorted(glob.glob(os.path.join(clip_dir, "event_clip_*.h5")))

print(f"Found {len(clip_paths)} clips")

for clip_path in clip_paths:
    process_event_clip_to_frames(
        clip_path=clip_path,
        time_windows=time_windows,
        output_dir=output_dir,
        width=WIDTH,
        height=HEIGHT
    )

Found 175 clips


KeyboardInterrupt: 